# 0.0 Set up

In [1]:
import os
import sys
import importlib
from pathlib import Path

import re
import numpy as np
import pandas as pd

# Change working directory to project directory
project_dir = str(Path.cwd().parent)
if 'modules' not in os.listdir(Path.cwd()):
    os.chdir(project_dir)

# 1.0 Data Preprocessing

## 1.1 Enrollment

In [102]:
from modules import enrollment_preprocessor

# Reload the module
importlib.reload(enrollment_preprocessor)

from modules import enrollment_preprocessor as ep

### 1.1.1 Module

In [103]:
# Initialize processor
processor = ep.EnrollmentDataProcessor()

In [104]:
%%time
# Process data
long_data = processor.process()

# Extract SHS offerings
shs_offerings = processor.extract_shs_offerings()

# Get summary
summary = processor.get_summary()
print("Data Summary:", summary)

INFO:modules.enrollment_preprocessor:Trimmed whitespaces from 99 string columns
INFO:modules.enrollment_preprocessor:Loaded 60167 records with 100 columns
INFO:modules.enrollment_preprocessor:Trimmed whitespaces from 17 string columns
INFO:modules.enrollment_preprocessor:Transformed to long format: 3489686 records


Data Summary: {'total_records': 3489686, 'total_enrollment': 26916754.0, 'unique_schools': 0, 'grade_levels': {'G11': 962672, 'G12': 962672, 'K': 120334, 'G1': 120334, 'G2': 120334, 'G3': 120334, 'G4': 120334, 'G5': 120334, 'G6': 120334, 'Elementary': 120334, 'G7': 120334, 'G8': 120334, 'G9': 120334, 'G10': 120334, 'JHS': 120334}, 'shs_offerings': {'ABM': 240668, 'HUMSS': 240668, 'STEM': 240668, 'GAS': 240668, 'PBM': 240668, 'TVL': 240668, 'SPORTS': 240668, 'ARTS & DESIGN': 240668}, 'gender_distribution': {'Male': 1744843, 'Female': 1744843}}
CPU times: user 24.5 s, sys: 2 s, total: 26.5 s
Wall time: 26.8 s


In [105]:
print(long_data.shape)
display(long_data.head())

(3489686, 20)


,region,division,district,school_id,school_name,street_address,province,municipality,legislative_district,barangay,sector,school_subclassification,school_type,modified_coc,enrollment_count,grade_level,gender,shs_offering,student_type,school_id_processed
0,Region I,Ilocos Norte,Bacarra I,100001,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,LIBTONG,Public,DepED Managed,School with no Annexes,Purely ES,4.0,K,Male,None,regular,100001
1,Region I,Ilocos Norte,Bacarra I,100002,Bacarra CES,Santa Rita,ILOCOS NORTE,BACARRA,1st District,SANTA RITA (POB.),Public,DepED Managed,School with no Annexes,Purely ES,26.0,K,Male,None,regular,100002
2,Region I,Ilocos Norte,Bacarra I,100003,Buyon ES,NONE,ILOCOS NORTE,BACARRA,1st District,BUYON,Public,DepED Managed,School with no Annexes,Purely ES,8.0,K,Male,None,regular,100003
3,Region I,Ilocos Norte,Bacarra I,100004,Ganagan Elementary School,"#37 Ganagan,Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,GANAGAN,Public,DepED Managed,School with no Annexes,Purely ES,9.0,K,Male,None,regular,100004
4,Region I,Ilocos Norte,Bacarra I,100005,Macupit ES,Macupit,ILOCOS NORTE,BACARRA,1st District,MACUPIT,Public,DepED Managed,School with no Annexes,Purely ES,5.0,K,Male,None,regular,100005


In [106]:
mask = long_data['gender'].isin(['Male','Female'])
df_enr = long_data.loc[mask].copy()
sum_enrollment = df_enr['enrollment_count'].sum()
display(sum_enrollment)

26916754.0

In [108]:
long_data.dtypes

region                              object
division                            object
district                            object
school_id                            int64
school_name                         object
street_address                      object
province                            object
municipality                        object
legislative_district                object
barangay                            object
sector                              object
school_subclassification            object
school_type                         object
modified_coc                        object
enrollment_count                   float64
grade_level                       category
gender                              object
shs_offering                        object
student_type                        object
school_id_processed         string[python]
dtype: object

In [64]:
# # Export processed data
# processor.export_processed()

### 1.1.2 Analog

In [9]:
# df_tmp = processor.raw_data.copy()
# df_tmp['School ID'] = df_tmp['School ID'].astype('string')

In [10]:
# df_tmp.columns

In [11]:
# df_tmp = df_tmp.rename(
#     columns={
#         'G11ACAD Male':'G11 ACAD Male',
#         'G11ACAD Female':'G11 ACAD Female',
#         'G12ACAD Male':'G12 ACAD Male',
#         'G12ACAD Female':'G12 ACAD Female',
#     }
# )

# cols_profiles = df_tmp.loc[:, "Region":"Modified COC"].columns
# cols_data = df_tmp.loc[:, "K Male":"SHS Total"].columns.tolist()
# pattern_exclude = r"total|to|jhs male|jhs female|g11 acad male|g11 acad female|g12 acad male|g12 acad female"
# cols_non_total = [col for col in cols_data if not re.search(pattern_exclude, col, flags=re.IGNORECASE)]

# diff_cols = set(cols_data).difference(set(cols_non_total))

# # To manually inspect check columns
# # display(cols_non_total)

# # To manually inspect dropped columns
# # display(diff_cols)

In [12]:
# pvt_tmp = df_tmp.melt(
#     id_vars="School ID",
#     value_vars=cols_non_total,
#     var_name="column_labels",
#     value_name="enrollment_count"
# )
# pvt_tmp['enrollment_count'] = pd.to_numeric(pvt_tmp['enrollment_count'], errors='coerce')
# pvt_tmp = pvt_tmp[pvt_tmp['enrollment_count'].notna()]

In [13]:
# pvt_tmp

In [14]:
# pvt_tmp['enrollment_count'].sum()

## 1.2. Public Coordinates

In [15]:
from modules import public_coordinates_processor

# Reload the module
importlib.reload(public_coordinates_processor)

from modules import public_coordinates_processor as scp

In [16]:
# Initialize processor
processor = scp.SchoolCoordinatesProcessor(verbose=False)

# Process data with automatic coordinate fixing
processed_data = processor.process()

# Get quality report
quality_report = processor.get_quality_report()
print("Data Quality Report:")
for key, value in quality_report.items():
    if isinstance(value, dict):
        print(f"  {key}: {len(value)} items")
    else:
        print(f"  {key}: {value}")

Data Quality Report:
  total_records: 47821
  missing_longitude: 632
  missing_latitude: 632
  missing_both: 632
  out_of_bounds: 73
  potentially_switched: 0
  valid_coordinates: 47116
  issues: []
  total_schools: 47821
  schools_with_valid_coords: 47116
  schools_with_missing_coords: 632
  coordinate_completeness_rate: 98.52575228456118
  regional_distribution: 17 items
  division_distribution: 10 items


In [17]:
processed_data.head(3)

,region,division,district,lis_school_id,nsbi_school_id,school_name,street_address,province,municipality,legislative_district,barangay,longitude,latitude,coord_valid,coord_missing,coord_out_of_bounds,coord_potentially_switched,school_id_processed
0,Region I,Ilocos Norte,Bacarra I,100001,100001.0,Apaleng-Libtong ES,"Brgy. 21, Libtong, Bacarra, Ilocos Norte",ILOCOS NORTE,BACARRA,1st District,LIBTONG,120.614372,18.266860,True,False,False,False,100001
1,Region I,Ilocos Norte,Bacarra I,100002,100002.0,Bacarra CES,Santa Rita,ILOCOS NORTE,BACARRA,1st District,SANTA RITA (POB.),120.609487,18.251272,True,False,False,False,100002
2,Region I,Ilocos Norte,Bacarra I,100003,100003.0,Buyon ES,NONE,ILOCOS NORTE,BACARRA,1st District,BUYON,120.616050,18.234670,True,False,False,False,100003


In [18]:
# Prepare merge-ready data
merge_data = processor.merge_ready_data()
print(f"\nMerge-ready data prepared with {len(merge_data)} records")


Merge-ready data prepared with 47821 records


In [19]:
print(merge_data.shape)
display(merge_data.head())

(47821, 14)


,lis_school_id,school_name,longitude,latitude,region,division,district,province,municipality,barangay,coord_valid,coord_missing,coord_out_of_bounds,coord_potentially_switched
school_id_processed,,,,,,,,,,,,,,
100001,100001,Apaleng-Libtong ES,120.614372,18.266860,Region I,Ilocos Norte,Bacarra I,ILOCOS NORTE,BACARRA,LIBTONG,True,False,False,False
100002,100002,Bacarra CES,120.609487,18.251272,Region I,Ilocos Norte,Bacarra I,ILOCOS NORTE,BACARRA,SANTA RITA (POB.),True,False,False,False
100003,100003,Buyon ES,120.616050,18.234670,Region I,Ilocos Norte,Bacarra I,ILOCOS NORTE,BACARRA,BUYON,True,False,False,False
100004,100004,Ganagan Elementary School,120.587415,18.250121,Region I,Ilocos Norte,Bacarra I,ILOCOS NORTE,BACARRA,GANAGAN,True,False,False,False
100005,100005,Macupit ES,120.641002,18.294093,Region I,Ilocos Norte,Bacarra I,ILOCOS NORTE,BACARRA,MACUPIT,True,False,False,False


## 1.3 Private School Coodinates

In [20]:
# from modules import private_schools_reader

# # Reload the module
# importlib.reload(private_schools_reader)

# from modules import private_schools_reader as pcr

In [21]:
# reader = pcr.PrivateSchoolsReader("data/private/raw_validation_sheets")
# data = reader.read_all_files()
# raw_data = reader.get_raw_data()

In [22]:
# raw_data.keys()

In [23]:
# len(raw_data.keys())

In [24]:
# raw_data['CAR - List of  Private School in the Philippines - DONE'].keys()

### 1.3.1 Preferred processing and collation of Excel files

In [25]:
# all_region_dfs = []
# for fname, divisions_dict in raw_data.items():
#     # print("*"+"="*30+fname+"="*30+"*")
#     # print(fname)
#     # display(divisions_dict.keys())

#     processed_divisions_dfs = []
#     for division_name, df_div in divisions_dict.items():
#         # print(division_name)
#         # print(df_div.shape)
      
#         # Detect row where "Region" appears in any of the first three columns
#         # (use fewer if the DF has <3 columns)
#         max_search_cols = min(3, df_div.shape[1])
        
#         region_row_mask = (
#             df_div.iloc[:, :max_search_cols]
#               .astype(str)
#               .apply(lambda s: s.str.strip().str.fullmatch(r'(?i)region'))
#               .any(axis=1)
#         )
        
#         matches = region_row_mask[region_row_mask].index.tolist()
#         if not matches:
#             raise ValueError(f'No header row found with "Region" in first {max_search_cols} column(s) for sheet "{division_name}" in file "{fname}".')
        
#         region_index = matches[0]

#         # Preprocess headers
#         headers = df_div.iloc[region_index, :].values
#         headers_processed = ['_'.join(str(col).strip().lower().split(' ')) for col in headers]
#         df_div.columns = headers_processed

#         # Retain first instance of columns whose headers are duplicated
#         df_div = df_div.loc[:, ~df_div.columns.duplicated()]

#         # Drop irrelevant rows before "Region"
#         df_div = df_div.iloc[region_index+1:, :].copy()
#         df_div['beis_school_id'] = df_div['beis_school_id'].astype('string')

#         # Add additional columns to tag source file
#         df_div['excel_filename'] = fname
#         df_div['sheet_name'] = division_name
        
#         # display(df_div.head())
#         processed_divisions_dfs.append(df_div)
#         # break

#     processed_divisions_dfs = pd.concat(processed_divisions_dfs, ignore_index=True)
#     # print(processed_divisions_dfs.shape)
#     # display(processed_divisions_dfs.head())

#     all_region_dfs.append(processed_divisions_dfs)
#     # break
    
# all_region_dfs = pd.concat(all_region_dfs, ignore_index=True)
# all_region_dfs = all_region_dfs.loc[:, :"sheet_name"]
# print(all_region_dfs.shape)

In [26]:
# !ls '.'

In [27]:
# all_region_dfs.to_csv('output/processed_collated_private_coordinates.csv', index=False)

### 1.3.2 Module

In [28]:
from modules import private_coordinates_processor

# Reload the module
importlib.reload(private_coordinates_processor)

from modules import private_coordinates_processor as psr

In [29]:
%%time
# Fast processing with minimal logging
processor = psr.PrivateSchoolsProcessor(verbose=False)

# Process data efficiently
processed_data = processor.process()

# Get summary
summary = processor.get_summary()
# print(f"Processed {summary['total_files_processed']} files")
# print(f"Success rate: {summary['success_rate']:.1f}%")
# print(f"Final dataset: {summary['final_dataset_rows']} rows × {summary['final_dataset_columns']} columns")

CPU times: user 23.9 s, sys: 255 ms, total: 24.1 s
Wall time: 34.8 s


In [30]:
# Validate coordinates if data was processed
if len(processed_data) > 0:
    # Perform coordinate validation
    coord_summary = processor.validate_coordinates()

    if coord_summary:
        print(f"\nCoordinate Validation Results:")
        print(f"  Valid coordinates: {coord_summary['valid_coordinates']:,} ({coord_summary['validation_rate']:.1f}%)")
        print(f"  Invalid coordinates: {coord_summary['invalid_coordinates']:,}")

        if coord_summary['latitude_column'] and coord_summary['longitude_column']:
            print(f"  Columns used: {coord_summary['latitude_column']}, {coord_summary['longitude_column']}")

        if coord_summary['issues_found'] and coord_summary['issues_found'] != ['No major issues found']:
            print(f"  Issues found: {'; '.join(coord_summary['issues_found'])}")


Coordinate Validation Results:
  Valid coordinates: 10,206 (86.2%)
  Invalid coordinates: 1,631
  Columns used: latitude, longitude
  Issues found: 634 missing latitude values; 28 latitude values outside Philippine bounds; 648 missing longitude values; 480 longitude values in DMS format (not decimal degrees); 484 latitude values in DMS format (not decimal degrees); 284 longitude values in DMM format (not decimal degrees); 417 latitude values are non-numeric text; 286 latitude values in DMM format (not decimal degrees); 335 longitude values are non-numeric text; 46 longitude values outside Philippine bounds


In [31]:
processed_data.head(3)

,region,division,district,beis_school_id,school_name,street_address,mother_school_id,province,municipality,legislative_district,barangay,sector,school_subclassification,modified_coc,latitude,longitude,excel_filename,sheet_name,coordinates_valid
0,CAR,Abra,Bangued East,400860,Data Center College of the Philippines,"Brgy. Ubbog, Lipcan, Bangued, Abra",NaN,ABRA,BANGUED (Capital),Lone District,LIPCAN,Private,Non-Sectarian,Purely SHS,17.58521,120.6149,CAR - List of Private School in the Philippin...,Abra,True
1,CAR,Abra,Bangued East,406102,Abra Valley Colleges,McKinley Street Zone 4 Bangued Abra,NaN,ABRA,BANGUED (Capital),Lone District,ZONE 4 POB. (TOWN PROPER),Private,Non-Sectarian,All Offering,17.59657,120.6145,CAR - List of Private School in the Philippin...,Abra,True
2,CAR,Abra,Bangued East,406104,Divine Word College-Bangued,Rizal,NaN,ABRA,BANGUED (Capital),Lone District,ZONE 6 POB. (SINAPANGAN),Private,Non-Sectarian,JHS with SHS,17.59447,120.61364,CAR - List of Private School in the Philippin...,Abra,True


In [32]:
processed_data['coordinates_valid'].value_counts()

coordinates_valid
True     10206
False     1631
Name: count, dtype: int64

## 1.4 Public Seats

In [82]:
from modules import seat_learner_preprocessor

# Reload the module
importlib.reload(seat_learner_preprocessor)

from modules import seat_learner_preprocessor as slp

In [83]:
%%time
# Initialize processor
processor = slp.SeatLearnerProcessor()

# Process data to long format
long_data = processor.process()

# Get summary
summary = processor.get_summary()

INFO:modules.seat_learner_preprocessor:Starting seat-learner ratio data processing
INFO:modules.seat_learner_preprocessor:Loading data from data/public/SY 2023-2024 SEAT-LEARNER RATIO.xlsx
INFO:modules.seat_learner_preprocessor:Trimmed whitespaces from 13 string columns
INFO:modules.seat_learner_preprocessor:Loaded 47821 records with 26 columns
INFO:modules.seat_learner_preprocessor:Transforming seat data from wide to long format
INFO:modules.seat_learner_preprocessor:Trimmed whitespaces from 2 string columns
INFO:modules.seat_learner_preprocessor:Created long format data with 54360 records
INFO:modules.seat_learner_preprocessor:Education levels: ['Elementary', 'Junior High School', 'Senior High School']
INFO:modules.seat_learner_preprocessor:Processing completed successfully


CPU times: user 10.4 s, sys: 31.9 ms, total: 10.4 s
Wall time: 13.3 s


In [85]:
long_data.head()

,school_id,education_level,seat_count
0,100001,Elementary,195
1,100002,Elementary,731
2,100003,Elementary,192
3,100004,Elementary,134
4,100005,Elementary,44


In [84]:
long_data.dtypes

school_id          string[python]
education_level            object
seat_count                  int64
dtype: object